# Linear Regression — Predicting a Movie's Production Cost

In this notebook we'll learn the basics of **Linear Regression** and use it to predict the **production cost** of a movie, using a made-up (imaginary) dataset.

## What is Linear Regression?
Linear Regression tries to draw the "best-fit line" through data, so it can predict a **number** (not a category) — for example, a price, a cost, or a score.

It works by finding the best weights (coefficients) for an equation like:

`cost = w1*duration + w2*num_actors + w3*marketing_budget + ... + b`

## Different ways to train a Linear Regression model
There's more than one way to find those "best weights." Here are the common ones:

1. **Ordinary Least Squares (OLS)** — solves for the best weights directly using math (this is what `LinearRegression` in scikit-learn uses). Fast and exact for smaller datasets.
2. **Gradient Descent (SGD)** — starts with random weights and slowly improves them step by step. Useful for very large datasets.
3. **Ridge Regression** — like OLS, but adds a penalty to stop the model from relying too heavily on any one feature (helps avoid overfitting).
4. **Lasso Regression** — similar to Ridge, but can shrink some feature weights all the way to zero (helps pick out the most important features).

We'll try **OLS**, **Ridge**, and **Gradient Descent (SGD)** below and compare them.

## Step 1: Import the libraries we need

In [ ]:
# For creating and handling our data
import pandas as pd
import numpy as np

# Tools to split data and build our models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# So our "random" numbers are the same every time we run this notebook
np.random.seed(42)

## Step 2: Create an imaginary movie dataset

Since we don't have a real dataset, we'll invent one! We'll imagine that a movie's **production cost** depends on:
- `duration_minutes` — how long the movie is
- `num_actors` — how many main actors are cast
- `marketing_budget` — how much is spent on marketing (in $1000s)
- `is_action` — whether it's an action movie (1) or not (0), action movies tend to cost more

We'll make up a formula, then add some random "noise" so it feels realistic (real life is never a perfectly straight line).

In [ ]:
# Number of imaginary movies to create
n_movies = 300

# Make up random feature values
duration_minutes = np.random.randint(80, 180, n_movies)
num_actors = np.random.randint(2, 12, n_movies)
marketing_budget = np.random.randint(50, 2000, n_movies)   # in $1000s
is_action = np.random.randint(0, 2, n_movies)               # 0 = no, 1 = yes

# Make up a "true" relationship for production cost (in $1000s)
# Then add random noise so it's not a perfectly straight line
noise = np.random.normal(0, 300, n_movies)
production_cost = (
    15 * duration_minutes +
    120 * num_actors +
    0.8 * marketing_budget +
    1500 * is_action +
    2000 +
    noise
)

# Put everything into a DataFrame (our imaginary dataset)
movies = pd.DataFrame({
    'duration_minutes': duration_minutes,
    'num_actors': num_actors,
    'marketing_budget': marketing_budget,
    'is_action': is_action,
    'production_cost': production_cost.round(0)
})

movies.head()

## Step 3: Explore the data

Let's take a quick look at the shape and statistics of our imaginary dataset.

In [ ]:
# Basic stats: mean, min, max, etc. for each column
movies.describe()

## Step 4: Select features (X) and target (y)

- `X` = the columns we use to predict
- `y` = the column we're trying to predict (`production_cost`)

In [ ]:
X = movies.drop(columns=['production_cost'])
y = movies['production_cost']

print("Features:", list(X.columns))

## Step 5: Split into training and testing sets

We train on 80% of the movies and test on the remaining 20% (movies the model has never seen).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training movies:", len(X_train))
print("Testing movies:", len(X_test))

## Step 6: Scale the features

Gradient Descent works much better when all features are on a similar scale (e.g., `marketing_budget` ranges in the thousands, but `is_action` is just 0 or 1).
`StandardScaler` rescales every feature to have a mean of 0 and a standard deviation of 1.

In [ ]:
scaler = StandardScaler()

# Learn the scaling from the training data, then apply it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the SAME scaling to the test data (don't re-learn it)
X_test_scaled = scaler.transform(X_test)

## Step 7: Train three different Linear Regression models

We'll train and compare:
1. **OLS** (`LinearRegression`)
2. **Ridge Regression** (`Ridge`)
3. **Gradient Descent** (`SGDRegressor`)

In [ ]:
# 1. Ordinary Least Squares
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

# 2. Ridge Regression (alpha controls how strong the penalty is)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

# 3. Gradient Descent Regression
sgd_model = SGDRegressor(max_iter=1000, random_state=42)
sgd_model.fit(X_train_scaled, y_train)

print("All three models trained!")

## Step 8: Compare the models

We'll check two scores for each model:
- **MAE (Mean Absolute Error)**: on average, how far off our predictions are (in $1000s) — lower is better
- **R² score**: how much of the pattern in the data our model explains — closer to 1 is better

In [ ]:
models = {
    'OLS (LinearRegression)': ols_model,
    'Ridge': ridge_model,
    'Gradient Descent (SGD)': sgd_model
}

# Predict and score each model, then collect results in a table
results = []
for name, model in models.items():
    predictions = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    results.append({'Model': name, 'MAE ($1000s)': round(mae, 1), 'R2 Score': round(r2, 3)})

pd.DataFrame(results)

## Step 9: Try predicting a brand new imaginary movie

Let's invent one new movie and see what our OLS model predicts for its production cost.

In [ ]:
# A new imaginary action movie: 140 minutes, 6 main actors, $800k marketing budget
new_movie = pd.DataFrame({
    'duration_minutes': [140],
    'num_actors': [6],
    'marketing_budget': [800],
    'is_action': [1]
})

# Scale it the same way we scaled the training data
new_movie_scaled = scaler.transform(new_movie)

# Predict using our OLS model
predicted_cost = ols_model.predict(new_movie_scaled)
print(f"Predicted production cost: ${predicted_cost[0]:,.0f} thousand")

## Recap

In this notebook we:
1. Learned what Linear Regression is and a few ways to train it (OLS, Ridge, Gradient Descent)
2. Built our own imaginary movie dataset
3. Trained and compared three different linear regression models
4. Used our model to predict the cost of a brand new movie

### Ideas to try next:
- Change the formula in Step 2 to invent your own unique dataset
- Add more features, like `num_special_effects` or `is_sequel`
- Try `Lasso` (another regularized linear model) and compare it too